# Local HEALPix FFT: projection and reconstruction tests

This notebook validates the gnomonic projection, the fast FFT/IFFT round trip, pole handling, the 0/360-degree meridian and PyTorch autograd. The HEALPix-to-grid reprojection is bilinear, so the final reconstruction is expected to be approximate.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import healpix_geo

from healpix_analyse.fft_local import LocalFFT, fft, ifft

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
torch.manual_seed(0)

In [ ]:
def make_patch(lon_deg, lat_deg, *, level=7, radius_deg=5.0):
    cell_ids, _, _ = healpix_geo.nested.cone_coverage(
        (lon_deg, lat_deg), radius_deg, level, ellipsoid="sphere"
    )
    return np.asarray(cell_ids, dtype=np.int64)


def relative_rms(reference, estimate):
    reference = torch.as_tensor(reference)
    estimate = torch.as_tensor(estimate)
    return (
        torch.linalg.vector_norm(estimate - reference)
        / torch.linalg.vector_norm(reference)
    ).item()

## Geometry and FFT/IFFT reconstruction

The four cases exercise a regular equatorial patch, a patch crossing the longitude discontinuity, and both poles.

In [ ]:
cases = {
    "equator": (45.0, 0.0),
    "meridian_0": (359.0, 20.0),
    "north_pole": (0.0, 90.0),
    "south_pole": (0.0, -90.0),
}

results = []
transforms = {}
for name, centre in cases.items():
    cell_ids = make_patch(*centre)
    transform = LocalFFT(
        cell_ids, level=7, device=device, dtype=torch.float64
    )
    transforms[name] = transform

    # A smooth signal defined directly in the local tangent plane.
    x = transform.projected_x
    y = transform.projected_y
    data = torch.sin(25.0 * x) + 0.4 * torch.cos(17.0 * y)
    spectrum = transform.fft(data)
    reconstructed = transform.ifft(spectrum)

    constant = torch.ones_like(data)
    constant_reconstructed = transform.ifft(transform.fft(constant))
    results.append({
        "case": name,
        "cells": transform.n_cells,
        "grid": transform.grid_size,
        "radius_deg": transform.patch_radius_deg,
        "smooth_rel_rms": relative_rms(data, reconstructed),
        "constant_max_abs": (constant_reconstructed - constant).abs().max().item(),
    })

for result in results:
    print(result)

In [ ]:
# Visual inspection of the most singular geographic case.
transform = transforms["north_pole"]
data = torch.sin(25.0 * transform.projected_x) + 0.4 * torch.cos(17.0 * transform.projected_y)
grid = transform.project(data).detach().cpu()
reconstructed = transform.ifft(transform.fft(data)).detach().cpu()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(transform.projected_x.cpu(), transform.projected_y.cpu(), c=data.cpu(), s=8)
axes[0].set_title("HEALPix centres in tangent plane")
axes[0].set_aspect("equal")
axes[1].imshow(grid, origin="lower", cmap="viridis")
axes[1].set_title("Projected square grid")
axes[2].scatter(data.cpu(), reconstructed, s=8, alpha=0.6)
limits = [float(data.min().cpu()), float(data.max().cpu())]
axes[2].plot(limits, limits, "k--")
axes[2].set_xlabel("input")
axes[2].set_ylabel("reconstructed")
axes[2].set_title("FFT/IFFT round trip")
plt.tight_layout()

## Batched tensors, CUDA and autograd

In [ ]:
transform = transforms["equator"].to(device)
batch = torch.randn(3, transform.n_cells, device=device, dtype=torch.float64, requires_grad=True)
spectrum = transform.fft(batch)
reconstructed = transform.ifft(spectrum)
loss = reconstructed.square().mean()
loss.backward()

print("batch spectrum shape:", tuple(spectrum.shape))
print("batch reconstruction shape:", tuple(reconstructed.shape))
print("finite gradient:", bool(torch.isfinite(batch.grad).all()))
print("gradient norm:", batch.grad.norm().item())

## Functional API and radius guard

In [ ]:
cell_ids = make_patch(10.0, 30.0, level=7, radius_deg=4.0)
data = np.random.default_rng(0).standard_normal(len(cell_ids))
spectrum, transform = fft(
    cell_ids, 7, data, return_transform=True, device=device
)
reconstructed = ifft(spectrum, transform)
print("functional API relative RMS:", relative_rms(data, reconstructed))

large_patch = make_patch(0.0, 0.0, level=6, radius_deg=12.0)
try:
    LocalFFT(large_patch, level=6, max_patch_radius_deg=10.0, device=device)
except ValueError as error:
    print("expected radius error:", error)
else:
    raise AssertionError("A patch larger than 10 degrees should be rejected")